# KernelPack RAG — Qdrant Migration

Migration validation for the v3 retrieval pipeline from ChromaDB export to Qdrant. This notebook runs the standalone migration, verifies every migrated point against `chroma_export.pkl`, rehydrates the ChromaDB v3 baseline, and compares backend and pipeline retrieval on the same solver QA set.

## Key Findings

| Check | Comparison | recall@3 | recall@5 | recall@10 | Finding |
|-------|------------|----------|----------|-----------|---------|
| Backend parity | ChromaDB dense vs Qdrant dense | 3/10 (30%) vs 3/10 (30%) | 5/10 (50%) vs 5/10 (50%) | 8/10 (80%) vs 8/10 (80%) | Dense recall matches, and ordered IDs match for every solver QA query at k=3, 5, and 10. |
| Hybrid pipeline | ChromaDB v3 hybrid vs Qdrant hybrid | 8/10 (80%) vs 8/10 (80%) | 10/10 (100%) vs 10/10 (100%) | 10/10 (100%) vs 10/10 (100%) | Qdrant can reproduce the 02b-style hybrid retrieval path when dense parity holds and BM25+RRF are mirrored. |
| Reranker context | Qdrant dense vs Qdrant reranked | 3/10 (30%) vs 8/10 (80%) | 5/10 (50%) vs 8/10 (80%) | 8/10 (80%) vs 10/10 (100%) | Cross-encoder reranking improves dense-only Qdrant retrieval, but it is a downstream experiment, not migration evidence. |

## Next Steps

- Keep dense ordered-ID parity as the gate for future storage changes; the backend parity result is what makes the hybrid comparison interpretable.
- Move from notebook-local BM25+RRF to a production Qdrant hybrid implementation only after preserving the same k=3/5/10 behavior against this notebook.
- Evaluate the cross-encoder reranker as v4, motivated by the reranker-context gains, but report it separately from migration validation.


## Setup

Install dependencies and configure paths.

In [80]:
%pip install -q qdrant-client



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [81]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd()
if (ROOT / "experiments").exists():
    EXPERIMENTS_DIR = ROOT / "experiments"
else:
    EXPERIMENTS_DIR = ROOT
    ROOT = EXPERIMENTS_DIR.parent

EXPORT_PATH = EXPERIMENTS_DIR / "chroma_export.pkl"
MIGRATION_SCRIPT = EXPERIMENTS_DIR / "scripts" / "migrate_to_qdrant.py"
QDRANT_STORAGE_PATH = ROOT / "qdrant_storage"
COLLECTION_NAME = "kernelpack-v3"

if str(EXPERIMENTS_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS_DIR))

print(f"Export:     {EXPORT_PATH}")
print(f"Script:     {MIGRATION_SCRIPT}")
print(f"Qdrant dir: {QDRANT_STORAGE_PATH}")


Export:     /Users/jordanchambers/public-projects/scientific-codebase-rag/experiments/chroma_export.pkl
Script:     /Users/jordanchambers/public-projects/scientific-codebase-rag/experiments/scripts/migrate_to_qdrant.py
Qdrant dir: /Users/jordanchambers/public-projects/scientific-codebase-rag/qdrant_storage


## Run Migration Script

Run the standalone migration script, then verify every exported point in Qdrant. The verification checks stable point IDs, payload text, metadata fields, and vectors within a 1e-5 tolerance.


In [82]:
if "qdrant" in globals():
    qdrant.close()
    del qdrant

cmd = [
    sys.executable,
    str(MIGRATION_SCRIPT),
    "--export-path", str(EXPORT_PATH),
    "--storage-path", str(QDRANT_STORAGE_PATH),
    "--collection-name", COLLECTION_NAME,
]

result = subprocess.run(cmd, text=True, capture_output=True)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()


Upserted 100/187
Upserted 187/187
Migration complete: 187 points in Qdrant collection 'kernelpack-v3'



In [83]:
from scripts.migrate_to_qdrant import load_chroma_export, make_qdrant_client, stable_point_id

verification_export = load_chroma_export(EXPORT_PATH)
verification_qdrant = make_qdrant_client(storage_path=QDRANT_STORAGE_PATH)

try:
    expected_point_ids = [stable_point_id(source_id) for source_id in verification_export.ids]
    duplicate_count = len(expected_point_ids) - len(set(expected_point_ids))
    assert duplicate_count == 0, f"Stable Qdrant point ID collision count: {duplicate_count}"

    batch_size = 64
    for start in range(0, verification_export.count, batch_size):
        end = min(start + batch_size, verification_export.count)
        source_ids = verification_export.ids[start:end]
        point_ids = expected_point_ids[start:end]
        documents = verification_export.documents[start:end]
        metadatas = verification_export.metadatas[start:end]
        embeddings = verification_export.embeddings[start:end]

        points = verification_qdrant.retrieve(
            collection_name=COLLECTION_NAME,
            ids=point_ids,
            with_payload=True,
            with_vectors=True,
        )
        points_by_id = {point.id: point for point in points}
        missing = [source_id for source_id, point_id in zip(source_ids, point_ids) if point_id not in points_by_id]
        assert not missing, f"Missing Qdrant points for exported IDs: {missing[:10]}"

        for source_id, point_id, document, metadata, embedding in zip(
            source_ids, point_ids, documents, metadatas, embeddings
        ):
            point = points_by_id[point_id]
            payload = point.payload or {}

            assert payload.get("original_id") == source_id, (
                f"Payload original_id mismatch for {source_id}: "
                f"got {payload.get('original_id')!r}"
            )
            assert payload.get("text") == document, f"Payload text mismatch for {source_id}"

            for key, expected_value in metadata.items():
                actual_value = payload.get(key)
                assert actual_value == expected_value, (
                    f"Payload metadata mismatch for {source_id}: key {key!r}; "
                    f"expected {expected_value!r}, got {actual_value!r}"
                )

            vector = point.vector
            if isinstance(vector, dict):
                assert len(vector) == 1, f"Unexpected named-vector payload for {source_id}: {vector.keys()}"
                vector = next(iter(vector.values()))
            assert vector is not None, f"Missing vector for {source_id}"
            assert len(vector) == len(embedding), (
                f"Vector dimension mismatch for {source_id}: "
                f"expected {len(embedding)}, got {len(vector)}"
            )
            max_abs_diff = max(abs(float(actual) - float(expected)) for actual, expected in zip(vector, embedding))
            assert max_abs_diff <= 1e-5, (
                f"Vector mismatch for {source_id}: max absolute difference {max_abs_diff:.6g}"
            )
finally:
    verification_qdrant.close()

print(f"Verified {verification_export.count} Qdrant points against exported documents, metadata, and vectors.")


Verified 187 Qdrant points against exported documents, metadata, and vectors.


## Inspect Qdrant Collection

Inspect collection-level counts and print a small scroll sample of migrated payloads. This is a quick readability check after the exhaustive point-by-point verification above.


In [84]:
from scripts.migrate_to_qdrant import load_chroma_export, make_qdrant_client

export = load_chroma_export(EXPORT_PATH)
qdrant = make_qdrant_client(storage_path=QDRANT_STORAGE_PATH)
collection = qdrant.get_collection(COLLECTION_NAME)

print(f"Export chunks: {export.count}")
print(f"Qdrant points: {collection.points_count}")
print(f"Vector dim:    {export.vector_dim}")
assert collection.points_count == export.count, "COUNT MISMATCH - do not proceed"


Export chunks: 187
Qdrant points: 187
Vector dim:    768


In [85]:
# Spot-check: verify a specific solver chunk migrated correctly with expected payload fields.
from qdrant_client.models import Filter, FieldCondition, MatchValue

points, _ = qdrant.scroll(
    collection_name=COLLECTION_NAME,
    scroll_filter=Filter(
        must=[
            FieldCondition(key="function_name", match=MatchValue(value="solve")),
            FieldCondition(key="class_name",    match=MatchValue(value="NonlinearVariablePoissonSolver"))
        ]
    ),
    with_payload=True,
    with_vectors=False,
    limit=5
)

for p in points:
    print(p.id)
    print(p.payload)

7353574893483140942
{'end_line': 344, 'function_name': 'solve', 'start_line': 255, 'class_name': 'NonlinearVariablePoissonSolver', 'file_path': 'solvers/nonlinear_variable_poisson.py', 'text': 'Solves the nonlinear variable-coefficient Poisson problem with Newton iterations, GMRES linear solves, and a simple backtracking line search. It returns the converged solution, operators, boundary data, iteration count, and residual norm, or raises if convergence fails.\n\ndef solve(\n        self,\n        forcing: Callable[..., np.ndarray] | np.ndarray,\n        forcing_u: Callable[..., np.ndarray] | np.ndarray,\n        coeff: Callable[..., np.ndarray] | np.ndarray,\n        coeff_u: Callable[..., np.ndarray] | np.ndarray,\n        neu_coeff_func: Callable[..., np.ndarray] | np.ndarray,\n        dir_coeff_func: Callable[..., np.ndarray] | np.ndarray,\n        bc: Callable[..., np.ndarray] | np.ndarray,\n        initial_guess: np.ndarray | None = None,\n    ) -> dict[str, object]:\n        if 

In [86]:
sample_points, _ = qdrant.scroll(
    collection_name=COLLECTION_NAME,
    limit=5,
    with_payload=True,
    with_vectors=False,
)

for point in sample_points:
    payload = point.payload
    symbol = payload.get("function_name") or "<module>"
    class_name = payload.get("class_name")
    if class_name:
        symbol = f"{class_name}.{symbol}"
    print(f"{point.id} | {payload.get('original_id')} | {symbol}")


39588040637615472 | solvers/variable_poisson.py:22-28 | build_variable_pde_operator
51310359373432655 | solvers/diffusion.py:93-97 | DiffusionSolver.set_step_size
71467843054861549 | solvers/incompressible_euler.py:156-173 | PUSLIncompressibleEulerSolver.bdf3_step
99181745899189812 | solvers/pu_sl_advection.py:526-559 | sample_backward_traced_values
123999120490094043 | solvers/detail/incompressible_euler_bdf_backend.py:53-64 | _boundary_lambda_matrix


## Rehydrate ChromaDB v3 Baseline

Rebuild the ChromaDB baseline from the exported v3 documents, metadata, IDs, and embeddings. BM25 is rebuilt with the same identifier-aware tokenizer used in 02b, and the helper retrievers below define dense and hybrid paths for both backends.


In [87]:
import chromadb
from rank_bm25 import BM25Okapi
from utils.chunking import bm25_tokenize
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("microsoft/unixcoder-base")
model.max_seq_length = 512
ef = SentenceTransformerEmbeddingFunction(model_name="microsoft/unixcoder-base")
ef._model = model

chroma_client = chromadb.EphemeralClient()
CHROMA_BASELINE_COLLECTION = "kp-hybrid-v3-rehydrated"
try:
    chroma_client.delete_collection(CHROMA_BASELINE_COLLECTION)
except Exception:
    pass

chroma_col = chroma_client.get_or_create_collection(
    CHROMA_BASELINE_COLLECTION,
    embedding_function=ef,
)
embeddings = export.embeddings.tolist() if hasattr(export.embeddings, "tolist") else export.embeddings
chroma_col.add(
    ids=export.ids,
    embeddings=embeddings,
    documents=export.documents,
    metadatas=export.metadatas,
)
bm25 = BM25Okapi([bm25_tokenize(doc) for doc in export.documents])

print(f"ChromaDB baseline collection: {chroma_col.count()} chunks")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 47815.45it/s]


ChromaDB baseline collection: 187 chunks


In [88]:
def embed_query(query: str) -> list[float]:
    """Use the same ChromaDB embedding function configured in the baseline notebook."""
    return ef([query])[0]


def as_result(chunk_id: str, document: str, metadata: dict, score: float, score_kind: str) -> dict:
    payload = dict(metadata)
    payload["text"] = document
    payload["original_id"] = chunk_id
    payload["score"] = float(score)
    payload["score_kind"] = score_kind
    return payload


def retrieve_chromadb_dense(query: str, k: int = 10) -> list[dict]:
    """Dense-only ChromaDB retrieval for backend parity with Qdrant."""
    result = chroma_col.query(
        query_texts=[query],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )

    return [
        as_result(chunk_id, document, metadata, distance, "cosine distance; lower is better")
        for chunk_id, document, metadata, distance in zip(
            result["ids"][0],
            result["documents"][0],
            result["metadatas"][0],
            result["distances"][0],
        )
    ]


def retrieve_qdrant_dense(query: str, k: int = 10) -> list[dict]:
    """Dense-only Qdrant retrieval using the same query embedding as ChromaDB."""
    query_vector = embed_query(query)
    hits = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=k,
    ).points

    results = []
    for hit in hits:
        payload = dict(hit.payload)
        payload["score"] = float(hit.score)
        payload["score_kind"] = "cosine similarity; higher is better"
        results.append(payload)
    return results


def retrieve_chromadb_v3_hybrid(query: str, k: int = 10) -> list[dict]:
    """Rehydrated v3 baseline: ChromaDB dense retrieval + BM25 with RRF fusion."""
    dense_res = chroma_col.query(query_texts=[query], n_results=10)
    dense_ids = dense_res["ids"][0]

    bm25_scores = bm25.get_scores(bm25_tokenize(query))
    top_bm25 = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:10]
    bm25_ids = [export.ids[i] for i in top_bm25]

    rrf = {}
    for rank, chunk_id in enumerate(dense_ids):
        rrf[chunk_id] = rrf.get(chunk_id, 0.0) + 1 / (rank + 60)
    for rank, chunk_id in enumerate(bm25_ids):
        rrf[chunk_id] = rrf.get(chunk_id, 0.0) + 1 / (rank + 60)

    top_ids = sorted(rrf, key=rrf.__getitem__, reverse=True)[:k]
    fetched = chroma_col.get(ids=top_ids, include=["documents", "metadatas"])
    id_to_idx = {chunk_id: i for i, chunk_id in enumerate(fetched["ids"])}

    results = []
    for chunk_id in top_ids:
        idx = id_to_idx[chunk_id]
        results.append(
            as_result(
                chunk_id,
                fetched["documents"][idx],
                fetched["metadatas"][idx],
                rrf[chunk_id],
                "RRF score; higher is better",
            )
        )
    return results

def retrieve_qdrant_hybrid(query: str, k: int = 10) -> list[dict]:
    """Qdrant hybrid: dense + BM25 with RRF fusion, mirroring 02b."""
    # dense leg
    query_vector = embed_query(query)
    dense_hits = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=10,
    ).points
    dense_ids = [h.payload["original_id"] for h in dense_hits]

    # BM25 leg (same tokenizer as 02b)
    bm25_scores = bm25.get_scores(bm25_tokenize(query))
    top_bm25 = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:10]
    bm25_ids = [export.ids[i] for i in top_bm25]

    # RRF fusion
    rrf = {}
    for rank, cid in enumerate(dense_ids):
        rrf[cid] = rrf.get(cid, 0.0) + 1 / (rank + 60)
    for rank, cid in enumerate(bm25_ids):
        rrf[cid] = rrf.get(cid, 0.0) + 1 / (rank + 60)

    top_ids = sorted(rrf, key=rrf.__getitem__, reverse=True)[:k]
    fetched = chroma_col.get(ids=top_ids, include=["documents", "metadatas"])
    id_to_idx = {cid: i for i, cid in enumerate(fetched["ids"])}

    results = []
    for cid in top_ids:
        idx = id_to_idx[cid]
        results.append(as_result(
            cid,
            fetched["documents"][idx],
            fetched["metadatas"][idx],
            rrf[cid],
            "RRF score; higher is better",
        ))
    return results

## Evaluation Helpers

Load the same solver QA set used in 02b and define the recall-table helpers. The hit test is imported from `utils.eval_utils` so symbol matching stays consistent with the baseline notebook.


In [89]:
from utils.eval_utils import is_hit

qa_path = EXPERIMENTS_DIR / "qa_pairs" / "solvers_qa.json"
qa_pairs_solvers = json.loads(qa_path.read_text())
print(f"Loaded {len(qa_pairs_solvers)} solver QA pairs")


def result_ids(results: list[dict]) -> list[str]:
    return [result["original_id"] for result in results]


def symbol_name(payload: dict) -> str:
    function_name = payload.get("function_name") or "<module>"
    class_name = payload.get("class_name") or ""
    return f"{class_name}.{function_name}" if class_name else function_name


def run_eval(retriever, qa_pairs: list[dict], k: int) -> dict:
    hits = {"api": 0, "workflow": 0, "conceptual": 0}
    totals = {"api": 0, "workflow": 0, "conceptual": 0}

    for qa in qa_pairs:
        results = retriever(qa["query"], k=k)
        totals[qa["tier"]] += 1
        if any(is_hit(result, qa["source_symbol"]) for result in results):
            hits[qa["tier"]] += 1

    return {"hits": hits, "totals": totals}


def total_recall(result: dict) -> tuple[int, int, float]:
    total_hits = sum(result["hits"].values())
    total_pairs = sum(result["totals"].values())
    return total_hits, total_pairs, total_hits / total_pairs


def print_eval_table(title: str, rows: list[tuple[str, object]]) -> None:
    print(title)
    print("k  " + "  ".join(f"{label:>24}" for label, _ in rows))
    print("-  " + "  ".join("-" * 24 for _ in rows))
    for k in [3, 5, 10]:
        values = []
        for _, retriever in rows:
            hits, total, score = total_recall(run_eval(retriever, qa_pairs_solvers, k=k))
            values.append(f"{hits}/{total} ({score:.0%})".rjust(24))
        print(f"{k:<2} " + "  ".join(values))
    print()


Loaded 10 solver QA pairs


## Backend Parity: Dense vs Dense

This is the backend migration check. Both dense retrievers use the same exported documents, metadata, embeddings, query embedding function, and `k` values. The cell first prints recall, then asserts full ordered-ID parity for every solver QA query at k=3, 5, and 10; any mismatch prints a rank-by-rank diff and fails the notebook.


In [91]:
print_eval_table(
    "Backend parity recall: ChromaDB dense vs Qdrant dense",
    [
        ("ChromaDB dense", retrieve_chromadb_dense),
        ("Qdrant dense", retrieve_qdrant_dense),
    ],
)

backend_parity_failures = []
for qa_index, qa in enumerate(qa_pairs_solvers, 1):
    for k in [3, 5, 10]:
        chroma_ids = result_ids(retrieve_chromadb_dense(qa["query"], k=k))
        qdrant_ids = result_ids(retrieve_qdrant_dense(qa["query"], k=k))
        if chroma_ids != qdrant_ids:
            backend_parity_failures.append({
                "qa_index": qa_index,
                "qa": qa,
                "k": k,
                "chroma_ids": chroma_ids,
                "qdrant_ids": qdrant_ids,
            })

if backend_parity_failures:
    print("Backend parity ordered-ID mismatches found.")
    for failure in backend_parity_failures:
        qa = failure["qa"]
        chroma_ids = failure["chroma_ids"]
        qdrant_ids = failure["qdrant_ids"]
        print("=" * 88)
        print(f"QA {failure['qa_index']} | k={failure['k']} | target={qa['source_symbol']} | tier={qa['tier']}")
        print(f"Query: {qa['query']}")
        print("rank | ChromaDB dense ID | Qdrant dense ID | status")
        print("-----|-------------------|------------------|--------")
        for rank in range(1, max(len(chroma_ids), len(qdrant_ids)) + 1):
            chroma_id = chroma_ids[rank - 1] if rank <= len(chroma_ids) else "<missing>"
            qdrant_id = qdrant_ids[rank - 1] if rank <= len(qdrant_ids) else "<missing>"
            status = "OK" if chroma_id == qdrant_id else "DIFF"
            print(f"{rank:>4} | {chroma_id} | {qdrant_id} | {status}")
    raise AssertionError(
        f"Backend parity failed: {len(backend_parity_failures)} ordered ID list mismatch(es) "
        "between ChromaDB dense and Qdrant dense."
    )

print("Backend parity ordered-ID check passed for every solver QA query at k=3, 5, and 10.")


Backend parity recall: ChromaDB dense vs Qdrant dense
k            ChromaDB dense              Qdrant dense
-  ------------------------  ------------------------
3                3/10 (30%)                3/10 (30%)
5                5/10 (50%)                5/10 (50%)
10               8/10 (80%)                8/10 (80%)

Backend parity ordered-ID check passed for every solver QA query at k=3, 5, and 10.


## Pipeline Context: Hybrid vs Hybrid

After dense backend parity passes, compare the full 02b-style hybrid pipeline on both backends. Both paths use dense retrieval, identifier-aware BM25 tokenization, and RRF; equal recall here depends on the dense ordered-ID parity check above.


In [92]:
print_eval_table(
    "Pipeline comparison recall: ChromaDB v3 hybrid vs Qdrant hybrid",
    [
        ("ChromaDB v3 hybrid", retrieve_chromadb_v3_hybrid),
        ("Qdrant hybrid",      retrieve_qdrant_hybrid),
    ],
)

Pipeline comparison recall: ChromaDB v3 hybrid vs Qdrant hybrid
k        ChromaDB v3 hybrid             Qdrant hybrid
-  ------------------------  ------------------------
3                8/10 (80%)                8/10 (80%)
5              10/10 (100%)              10/10 (100%)
10             10/10 (100%)              10/10 (100%)



## Top-5 Qualitative Comparison

Print top-5 chunks for representative solver QA queries. The first diff check confirms ChromaDB dense and Qdrant dense have no top-5 differences; the second intentionally contrasts ChromaDB hybrid with Qdrant dense to show how BM25+RRF changes the pipeline relative to dense-only retrieval.


In [93]:
def print_top_chunks(label: str, results: list[dict], max_text_chars: int = 140) -> None:
    print(label)
    for rank, result in enumerate(results, 1):
        text = " ".join(result.get("text", "").split())[:max_text_chars]
        print(
            f"  {rank}. {result['original_id']} | "
            f"score={result['score']:.6f} ({result['score_kind']}) | "
            f"{symbol_name(result)}"
        )
        print(f"     {text}")


def first_result_difference(
    left_retriever,
    right_retriever,
    qa_pairs: list[dict],
    k: int = 5,
) -> dict | None:
    for qa in qa_pairs:
        left_ids = result_ids(left_retriever(qa["query"], k=k))
        right_ids = result_ids(right_retriever(qa["query"], k=k))
        if left_ids != right_ids:
            return {"qa": qa, "left_ids": left_ids, "right_ids": right_ids}
    return None


def print_difference(label: str, difference: dict | None, left_label: str, right_label: str) -> None:
    print(label)
    if difference is None:
        print("  No top-5 differences found across the solver QA set.")
        return

    qa = difference["qa"]
    left_ids = difference["left_ids"]
    right_ids = difference["right_ids"]
    left_only = [chunk_id for chunk_id in left_ids if chunk_id not in right_ids]
    right_only = [chunk_id for chunk_id in right_ids if chunk_id not in left_ids]

    print(f"  Query:  {qa['query']}")
    print(f"  Target: {qa['source_symbol']} ({qa['tier']})")
    print(f"  {left_label} ids:  {left_ids}")
    print(f"  {right_label} ids: {right_ids}")
    print(f"  {left_label}-only ids:  {left_only or 'same set; ranking differs'}")
    print(f"  {right_label}-only ids: {right_only or 'same set; ranking differs'}")


dense_difference = first_result_difference(
    retrieve_chromadb_dense,
    retrieve_qdrant_dense,
    qa_pairs_solvers,
    k=5,
)
pipeline_difference = first_result_difference(
    retrieve_chromadb_v3_hybrid,
    retrieve_qdrant_dense,
    qa_pairs_solvers,
    k=5,
)

print_difference(
    "Backend parity difference check: ChromaDB dense vs Qdrant dense",
    dense_difference,
    "ChromaDB dense",
    "Qdrant dense",
)
print()
print_difference(
    "Pipeline difference check: ChromaDB v3 hybrid vs Qdrant dense",
    pipeline_difference,
    "ChromaDB hybrid",
    "Qdrant dense",
)

if pipeline_difference is None and dense_difference is None:
    print("\nNo top-5 differences were found for either comparison on this QA set.")
elif pipeline_difference is not None:
    print("\nThe identified difference is a pipeline difference, not evidence of migration drift.")


Backend parity difference check: ChromaDB dense vs Qdrant dense
  No top-5 differences found across the solver QA set.

Pipeline difference check: ChromaDB v3 hybrid vs Qdrant dense
  Query:  For a publication benchmark with mixed Dirichlet and Neumann elliptic boundary data, how can I obtain the physical field together with the assembled matrix, right-hand side pieces, and nullspace metadata needed for reproducibility?
  Target: PoissonSolver.solve (api)
  ChromaDB hybrid ids:  ['solvers/_common.py:106-110', 'solvers/poisson.py:69-121', 'solvers/nonlinear_variable_poisson.py:246-253', 'solvers/detail/incompressible_euler_bdf_backend.py:181-185', 'solvers/incompressible_euler.py:13-22']
  Qdrant dense ids: ['solvers/_common.py:106-110', 'solvers/detail/incompressible_euler_bdf_backend.py:181-185', 'solvers/incompressible_euler.py:13-22', 'solvers/nonlinear_variable_poisson.py:144-165', 'solvers/detail/incompressible_euler_bdf_backend.py:14-23']
  ChromaDB hybrid-only ids:  ['solvers/po

In [94]:
sample_qas = []
anchor_difference = dense_difference or pipeline_difference
if anchor_difference is not None:
    sample_qas.append(anchor_difference["qa"])

for qa in qa_pairs_solvers:
    if len(sample_qas) >= 3:
        break
    if qa not in sample_qas:
        sample_qas.append(qa)

for sample_number, qa in enumerate(sample_qas, 1):
    print("=" * 88)
    print(f"Sample query {sample_number}: {qa['query']}")
    print(f"Target: {qa['source_symbol']} | tier={qa['tier']}")

    chroma_dense = retrieve_chromadb_dense(qa["query"], k=5)
    qdrant_dense = retrieve_qdrant_dense(qa["query"], k=5)
    chroma_hybrid = retrieve_chromadb_v3_hybrid(qa["query"], k=5)

    print_top_chunks("ChromaDB dense top-5", chroma_dense)
    print_top_chunks("Qdrant dense top-5", qdrant_dense)
    print_top_chunks("ChromaDB v3 hybrid top-5 (pipeline context)", chroma_hybrid)

    chroma_dense_ids = result_ids(chroma_dense)
    qdrant_dense_ids = result_ids(qdrant_dense)
    chroma_hybrid_ids = result_ids(chroma_hybrid)

    dense_overlap = sorted(set(chroma_dense_ids) & set(qdrant_dense_ids))
    pipeline_overlap = sorted(set(chroma_hybrid_ids) & set(qdrant_dense_ids))
    print(f"Dense ChromaDB/Qdrant top-5 overlap: {len(dense_overlap)}/5")
    print(f"Same ordered dense top-5: {chroma_dense_ids == qdrant_dense_ids}")
    print(f"Hybrid ChromaDB/Qdrant dense top-5 overlap: {len(pipeline_overlap)}/5")
    print(f"Same ordered hybrid-vs-dense top-5: {chroma_hybrid_ids == qdrant_dense_ids}")


Sample query 1: For a publication benchmark with mixed Dirichlet and Neumann elliptic boundary data, how can I obtain the physical field together with the assembled matrix, right-hand side pieces, and nullspace metadata needed for reproducibility?
Target: PoissonSolver.solve | tier=api
ChromaDB dense top-5
  1. solvers/_common.py:106-110 | score=0.586965 (cosine distance; lower is better) | build_system_rhs
     Concatenates the interior and boundary right-hand-side vectors for a Poisson solve. For pure Neumann systems it appends the zero constraint 
  2. solvers/detail/incompressible_euler_bdf_backend.py:181-185 | score=0.645583 (cosine distance; lower is better) | IncompressibleEulerBDFBackend.stationary_slip_wall
     Builds a slip-wall problem entry for the given one-based boundary indices with zero prescribed normal velocity. def stationary_slip_wall(bou
  3. solvers/incompressible_euler.py:13-22 | score=0.654259 (cosine distance; lower is better) | _make_physical_advection_domain

## Optional Cross-Encoder Reranking

The reranker is a downstream retrieval experiment. It is evaluated separately because it changes the Qdrant pipeline beyond the storage migration; the output compares Qdrant dense retrieval against Qdrant dense retrieval followed by cross-encoder reranking.


In [95]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Reranker loaded")


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9900.01it/s]


Reranker loaded


In [96]:
def retrieve_qdrant_hybrid_reranked(query: str, k: int = 10, fetch_k: int = 30) -> list[dict]:
    """Qdrant hybrid + cross-encoder reranker. fetch_k candidates from hybrid, rerank to k."""
    candidates = retrieve_qdrant_hybrid(query, k=fetch_k)  # ← hybrid, not dense
    pairs = [(query, candidate["text"]) for candidate in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, candidates), key=lambda item: item[0], reverse=True)

    results = []
    for score, candidate in ranked[:k]:
        payload = dict(candidate)
        payload["score"] = float(score)
        payload["score_kind"] = "cross-encoder score; higher is better"
        results.append(payload)
    return results

In [97]:
def retrieve_qdrant_reranked(query: str, k: int = 10, fetch_k: int = 30) -> list[dict]:
    candidates = retrieve_qdrant_dense(query, k=fetch_k)
    pairs = [(query, candidate["text"]) for candidate in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, candidates), key=lambda item: item[0], reverse=True)

    results = []
    for score, candidate in ranked[:k]:
        payload = dict(candidate)
        payload["score"] = float(score)
        payload["score_kind"] = "cross-encoder score; higher is better"
        results.append(payload)
    return results


print_eval_table(
    "Downstream experiment recall: Qdrant dense vs Qdrant reranked",
    [
        ("Qdrant dense", retrieve_qdrant_dense),
        ("Qdrant reranked", retrieve_qdrant_reranked),
    ],
)


Downstream experiment recall: Qdrant dense vs Qdrant reranked
k              Qdrant dense           Qdrant reranked
-  ------------------------  ------------------------
3                3/10 (30%)                8/10 (80%)
5                5/10 (50%)                8/10 (80%)
10               8/10 (80%)              10/10 (100%)



## Reranker on Hybrid Pipeline

Evaluate whether the cross-encoder improves recall when applied on top of hybrid retrieval (dense + BM25 + RRF), not just dense-only. This is a downstream experiment — not a migration validation.

In [98]:
print_eval_table(
    "Hybrid pipeline: Qdrant hybrid vs Qdrant hybrid+reranker",
    [
        ("Qdrant hybrid",          retrieve_qdrant_hybrid),
        ("Qdrant hybrid+reranker", retrieve_qdrant_hybrid_reranked),
    ],
)

Hybrid pipeline: Qdrant hybrid vs Qdrant hybrid+reranker
k             Qdrant hybrid    Qdrant hybrid+reranker
-  ------------------------  ------------------------
3                8/10 (80%)                8/10 (80%)
5              10/10 (100%)                8/10 (80%)
10             10/10 (100%)              10/10 (100%)



In [99]:
for qa in qa_pairs_solvers:
    results = retrieve_qdrant_hybrid(qa["query"], k=5)
    matched = any(
        f"{r.get('class_name', '')}.{r.get('function_name', '')}" == qa["source_symbol"]
        for r in results
    )
    if not matched:
        print(f"MISS: {qa['source_symbol']} | {qa['tier']}")
        print(f"  Query: {qa['query']}")

MISS: build_stencil_properties | conceptual
  Query: How are RBF-FD stencil size and polynomial degree chosen from a requested convergence order and differential operator order?
MISS: pu_patch_weight | conceptual
  Query: What compactly supported partition-of-unity weight does the localized patch machinery use for scientific interpolation and operators?


## Token Length Analysis

47.6% of chunks fall below 200 tokens. Spot check (cell above) confirms these are legitimate small helper functions — short length is a property of the KernelPack codebase, not a chunking deficiency. LLM summaries are prepended to all chunks. Recall@5 = 10/10 confirms retrieval is not impaired by chunk length.

In [100]:
from transformers import AutoTokenizer
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("microsoft/unixcoder-base")

# scroll your existing collection — check your notebook for the actual collection name
points, _ = qdrant.scroll("kernelpack-v3", with_payload=True, limit=200)

lengths = [(p.payload.get("function_name", "?"), 
            len(tokenizer(p.payload.get("text", ""), truncation=False)["input_ids"]))
           for p in points]

vals = [l for _, l in lengths]
print(f"p50={np.percentile(vals,50):.0f}  p90={np.percentile(vals,90):.0f}  max={max(vals)}")
print("Over 512:", len([(name, n) for name, n in lengths if n > 512]))
print("Over 450:", len([(name, n) for name, n in lengths if n > 450]))

p50=208  p90=514  max=1179
Over 512: 19
Over 450: 27


: 